In [1]:
%pip install opencv-python numpy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#stage 1 Optical flow, RANSAC, and texture transformation

In [6]:
from pathlib import Path

import cv2

import numpy as np

# Enter the video path
VIDEO_PATH = Path("C:/Users/user/Desktop/M2/Computer Vision and Image Processing/Major project/test2.mp4")

# Verify tracking without texture
TEXTURE_PATH = Path("C:/Users/user/Desktop/M2/Computer Vision and Image Processing/Major project/male.png")

# Once tracking is normal, change to the transparent PNG path
# TEXTURE_PATH = Path("male.png")

# Shi–Tomasi corner detection parameters
FEATURE_PARAMS = dict(
    maxCorners=150,
    qualityLevel=0.01,
    minDistance=7,
    blockSize=7
)

# Pyramid Lucas–Kanade optical flow parameters
LK_PARAMS = dict(
    winSize=(21, 21),
    maxLevel=3,
    criteria=(
        cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT,
        30,
        0.01
    )
)

# Forward-backward tracking error tolerance (in pixels)
FB_THRESHOLD = 1.5

# RANSAC reprojection error threshold (in pixels)
RANSAC_THRESHOLD = 3.0

# Minimum corresponding points required to maintain stable estimation
MIN_POINTS = 10

WINDOW_NAME = "Face tracking: Q quit | R reset | Space pause"

In [7]:
def initialize_tracking(frame):
    """Select the face in the current frame to create initial features and a reference bounding box."""

    selection_window = "Select face ROI, then press Enter"

    # Drag mouse to select face, press Enter to confirm, press Esc to cancel.
    x, y, w, h = cv2.selectROI(
        selection_window,
        frame,
        showCrosshair=True,
        fromCenter=False
    )
    cv2.destroyWindow(selection_window)

    if w == 0 or h == 0:
        raise ValueError("No valid face region selected.")

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Find feature points only within the selected area
    face_mask = np.zeros(gray.shape, dtype=np.uint8)
    face_mask[y:y + h, x:x + w] = 255

    points = cv2.goodFeaturesToTrack(
        gray,
        mask=face_mask,
        **FEATURE_PARAMS
    )

    if points is None or len(points) < MIN_POINTS:
        raise ValueError(
            "Too few features found in the face region. Please reselect or use a clearer video."
        )

    # Reference box corner order: Top-Left, Top-Right, Bottom-Right, Bottom-Left
    reference_corners = np.float32([
        [x, y],
        [x + w - 1, y],
        [x + w - 1, y + h - 1],
        [x, y + h - 1]
    ]).reshape(-1, 1, 2)

    return gray, points, reference_corners


def track_points(previous_gray, current_gray, previous_points,
                 reference_points):
    """Track optical flow while preserving the corresponding coordinates of each point in the initial frame."""

    # Forward: Previous frame → Current frame
    current_points, forward_status, _ = cv2.calcOpticalFlowPyrLK(
        previous_gray,
        current_gray,
        previous_points,
        None,
        **LK_PARAMS
    )

    if current_points is None or forward_status is None:
        return None, None

    # Remove points with failed forward tracking or invalid coordinates first
    forward_valid = (
        (forward_status.ravel() == 1)
        & np.isfinite(current_points.reshape(-1, 2)).all(axis=1)
    )

    if not np.any(forward_valid):
        return None, None

    previous_valid = previous_points[forward_valid]
    current_valid = current_points[forward_valid]
    reference_valid = reference_points[forward_valid]

    # Backward: Current frame → Previous frame
    backward_points, backward_status, _ = cv2.calcOpticalFlowPyrLK(
        current_gray,
        previous_gray,
        current_valid,
        None,
        **LK_PARAMS
    )

    if backward_points is None or backward_status is None:
        return None, None

    # Check if backward tracking returns to the original position
    fb_error = np.linalg.norm(
        previous_valid.reshape(-1, 2)
        - backward_points.reshape(-1, 2),
        axis=1
    )

    height, width = current_gray.shape
    xy = current_valid.reshape(-1, 2)

    # Current points must be within the image
    inside_frame = (
        (xy[:, 0] >= 0)
        & (xy[:, 0] < width)
        & (xy[:, 1] >= 0)
        & (xy[:, 1] < height)
    )

    reliable = (
        (backward_status.ravel() == 1)
        & np.isfinite(fb_error)
        & (fb_error < FB_THRESHOLD)
        & inside_frame
    )

    return current_valid[reliable], reference_valid[reliable]


def load_texture(texture_path):
    """Read BGRA texture; returns None if no path is provided."""

    if texture_path is None:
        return None

    texture = cv2.imread(str(texture_path), cv2.IMREAD_UNCHANGED)

    if texture is None:
        raise FileNotFoundError(f"Failed to read texture：{texture_path}")

    if texture.ndim != 3 or texture.shape[2] != 4:
        raise ValueError("Textures must be PNGs with an alpha channel (4-channel BGRA images).")

    if min(texture.shape[:2]) < 2:
        raise ValueError("Texture size is too small.")

    return texture


def overlay_texture(frame, texture, texture_to_frame):
    """Transform the texture and opacity, then overlay it onto the original video frame."""

    height, width = frame.shape[:2]

    texture_color = texture[:, :, :3].astype(np.float32)
    texture_alpha = texture[:, :, 3].astype(np.float32) / 255.0

    # Premultiply alpha to reduce black borders at transparent edges during interpolation.
    premultiplied_color = texture_color * texture_alpha[:, :, None]

    warped_color = cv2.warpPerspective(
        premultiplied_color,
        texture_to_frame,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )

    warped_alpha = cv2.warpPerspective(
        texture_alpha,
        texture_to_frame,
        (width, height),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0
    )

    output = (
        warped_color
        + frame.astype(np.float32) * (1.0 - warped_alpha[:, :, None])
    )

    return np.clip(output, 0, 255).astype(np.uint8)

In [8]:
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    cap.release()
    raise FileNotFoundError(f"Failed to open video：{VIDEO_PATH}")

success, first_frame = cap.read()

if not success:
    cap.release()
    raise RuntimeError("Failed to read the first frame of the video.")

try:
    texture = load_texture(TEXTURE_PATH)

    previous_gray, previous_points, reference_corners = (
        initialize_tracking(first_frame)
    )

    # Position of each tracking point in the initial frame
    reference_points = previous_points.copy()

    # Map the four corners of the texture to the initial face bounding box
    texture_to_reference = None

    if texture is not None:
        texture_height, texture_width = texture.shape[:2]

        texture_corners = np.float32([
            [0, 0],
            [texture_width - 1, 0],
            [texture_width - 1, texture_height - 1],
            [0, texture_height - 1]
        ])

        texture_to_reference = cv2.getPerspectiveTransform(
            texture_corners,
            reference_corners.reshape(4, 2)
        )

    fps = cap.get(cv2.CAP_PROP_FPS)

    if not np.isfinite(fps) or fps <= 0:
        fps = 30.0

    display_delay = max(1, int(round(1000.0 / fps)))

    print(f"Number of initial feature points：{len(previous_points)}")
    print("Initialization complete. Please run the next cell.")

except Exception:
    cap.release()
    cv2.destroyAllWindows()
    raise

Number of initial feature points：150
Initialization complete. Please run the next cell.


In [9]:
# Attempt to add points when valid tracking points fall below this number
REPLENISH_THRESHOLD = 80

# Minimum distance between new and existing points (in pixels)
NEW_POINT_DISTANCE = 10

# Whether to display bounding boxes and tracking points
SHOW_DEBUG = True

# Choose whether to save the video before running
save_answer = input("Save the output video? Enter 'y' to save, any other input to skip:")
SAVE_VIDEO = save_answer.strip().lower() == "y"

# Use a new filename for easy comparison with the homography version.
OUTPUT_PATH = VIDEO_PATH.with_name(
    VIDEO_PATH.stem + "_similarity_output.mp4"
)

writer = None
tracking_active = True

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

try:
    if SAVE_VIDEO:
        if OUTPUT_PATH.exists():
            overwrite_answer = input(
                f"{OUTPUT_PATH.name} File already exists. Overwrite? Enter 'y' to confirm:"
            )
            SAVE_VIDEO = overwrite_answer.strip().lower() == "y"

        if SAVE_VIDEO:
            frame_height, frame_width = first_frame.shape[:2]

            writer = cv2.VideoWriter(
                str(OUTPUT_PATH),
                cv2.VideoWriter_fourcc(*"mp4v"),
                fps,
                (frame_width, frame_height)
            )

            if not writer.isOpened():
                raise RuntimeError(
                    "Failed to create output video. Please check the path and codec."
                )

            print(f"Will be saved to:{OUTPUT_PATH}")

    while True:
        success, frame = cap.read()

        if not success:
            print("Video playback ended.")
            break

        # Always track on the original image.
        current_gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        output = frame.copy()

        if tracking_active:
            current_points, matched_reference = track_points(
                previous_gray,
                current_gray,
                previous_points,
                reference_points
            )

            if current_points is None or len(current_points) < MIN_POINTS:
                tracking_active = False

            else:
                # Similarity transform: translation, rotation, uniform scaling
                affine_matrix, ransac_status = (
                    cv2.estimateAffinePartial2D(
                        matched_reference.reshape(-1, 2),
                        current_points.reshape(-1, 2),
                        method=cv2.RANSAC,
                        ransacReprojThreshold=RANSAC_THRESHOLD,
                        maxIters=2000,
                        confidence=0.99,
                        refineIters=10
                    )
                )

                # Convert to a 3×3 matrix to maintain compatibility with existing texture and point-padding functions.
                homography = None

                if affine_matrix is not None:
                    homography = np.vstack([
                        affine_matrix,
                        np.array([[0.0, 0.0, 1.0]])
                    ])

                if (
                    homography is None
                    or ransac_status is None
                    or not np.isfinite(homography).all()
                ):
                    tracking_active = False

                else:
                    inlier_mask = ransac_status.ravel().astype(bool)

                    if np.count_nonzero(inlier_mask) < MIN_POINTS:
                        tracking_active = False

        if tracking_active:
            # Map the initial face bounding box to the current frame
            current_corners = cv2.perspectiveTransform(
                reference_corners,
                homography
            )

            corners_xy = current_corners.reshape(-1, 2)
            frame_height, frame_width = current_gray.shape

            # Check if the bounding box has obvious anomalies
            corners_valid = np.isfinite(corners_xy).all()

            if corners_valid:
                corners_valid = (
                    (np.abs(corners_xy[:, 0]) < 3 * frame_width).all()
                    and
                    (np.abs(corners_xy[:, 1]) < 3 * frame_height).all()
                )

            if corners_valid:
                contour = corners_xy.astype(np.float32)
                box_area = abs(cv2.contourArea(contour))

                corners_valid = (
                    cv2.isContourConvex(contour)
                    and box_area > 100
                    and box_area < frame_width * frame_height
                )

            if not corners_valid:
                tracking_active = False

        if tracking_active:
            polygon = np.rint(current_corners).astype(np.int32)

            # Overlay if texture exists
            if texture is not None:
                texture_to_frame = (
                    homography @ texture_to_reference
                )

                output = overlay_texture(
                    frame,
                    texture,
                    texture_to_frame
                )

            # Keep RANSAC inliers of the current frame
            kept_points = current_points[inlier_mask].copy()
            kept_reference = matched_reference[inlier_mask].copy()

            added_count = 0

            # Add points within the current tracking area when points are insufficient
            if len(kept_points) < REPLENISH_THRESHOLD:
                new_point_mask = np.zeros(
                    current_gray.shape,
                    dtype=np.uint8
                )

                cv2.fillConvexPoly(
                    new_point_mask,
                    polygon.reshape(-1, 2),
                    255
                )

                # Exclude areas near borders to reduce the chance of selecting background points
                new_point_mask = cv2.erode(
                    new_point_mask,
                    np.ones((9, 9), dtype=np.uint8),
                    iterations=1
                )

                # Exclude areas near existing tracking points
                for point in kept_points.reshape(-1, 2):
                    point_xy = tuple(
                        np.rint(point).astype(int)
                    )

                    cv2.circle(
                        new_point_mask,
                        point_xy,
                        NEW_POINT_DISTANCE,
                        0,
                        -1
                    )

                new_feature_params = FEATURE_PARAMS.copy()
                new_feature_params["maxCorners"] = (
                    FEATURE_PARAMS["maxCorners"]
                    - len(kept_points)
                )

                if new_feature_params["maxCorners"] > 0:
                    new_points = cv2.goodFeaturesToTrack(
                        current_gray,
                        mask=new_point_mask,
                        **new_feature_params
                    )

                    if new_points is not None:
                        try:
                            inverse_homography = np.linalg.inv(
                                homography
                            )

                            # Map the current new points back to the estimated reference frame position
                            new_reference = (
                                cv2.perspectiveTransform(
                                    new_points,
                                    inverse_homography
                                )
                            )

                            valid_new = np.isfinite(
                                new_reference.reshape(-1, 2)
                            ).all(axis=1)

                            new_points = new_points[valid_new]
                            new_reference = new_reference[valid_new]

                            added_count = len(new_points)

                            if added_count > 0:
                                kept_points = np.concatenate(
                                    [kept_points, new_points],
                                    axis=0
                                )

                                kept_reference = np.concatenate(
                                    [kept_reference, new_reference],
                                    axis=0
                                )

                        except np.linalg.LinAlgError:
                            # Skip adding points if the matrix cannot be inverted
                            pass

            if SHOW_DEBUG:
                # Tracking bounding box
                cv2.polylines(
                    output,
                    [polygon],
                    True,
                    (0, 255, 0),
                    2
                )

                # Green: Inliers; Red: Outliers
                for point, is_inlier in zip(
                    current_points.reshape(-1, 2),
                    inlier_mask
                ):
                    point_xy = tuple(
                        np.rint(point).astype(int)
                    )

                    color = (
                        (0, 255, 0)
                        if is_inlier
                        else (0, 0, 255)
                    )

                    cv2.circle(
                        output,
                        point_xy,
                        3,
                        color,
                        -1
                    )

                # Blue: New points added in this frame
                if added_count > 0:
                    for point in kept_points[-added_count:].reshape(-1, 2):
                        point_xy = tuple(
                            np.rint(point).astype(int)
                        )

                        cv2.circle(
                            output,
                            point_xy,
                            3,
                            (255, 0, 0),
                            -1
                        )

                label = (
                    f"Inliers: {np.count_nonzero(inlier_mask)} | "
                    f"Added: {added_count} | "
                    f"Total: {len(kept_points)}"
                )

                cv2.putText(
                    output,
                    label,
                    (20, 30),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.65,
                    (0, 255, 0),
                    2
                )

            # Update points for the next frame
            previous_points = kept_points
            reference_points = kept_reference

        if not tracking_active:
            cv2.putText(
                output,
                "Tracking lost. Press R to select face again.",
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.65,
                (0, 0, 255),
                2
            )

        # Each video frame is written only once
        if writer is not None:
            writer.write(output)

        cv2.imshow(WINDOW_NAME, output)
        key = cv2.waitKey(display_delay) & 0xFF

        # Space to pause, press Space again to resume
        if key == ord(" "):
            while True:
                key = cv2.waitKey(30) & 0xFF

                if key in (ord(" "), ord("q"), ord("r"), 27):
                    break

                if cv2.getWindowProperty(
                    WINDOW_NAME,
                    cv2.WND_PROP_VISIBLE
                ) < 1:
                    key = ord("q")
                    break

        # Q or Esc to exit
        if key in (ord("q"), 27):
            break

        if cv2.getWindowProperty(
            WINDOW_NAME,
            cv2.WND_PROP_VISIBLE
        ) < 1:
            break

        # R: Reselect the face in the current frame
        if key == ord("r"):
            try:
                (
                    current_gray,
                    previous_points,
                    reference_corners
                ) = initialize_tracking(frame)

                reference_points = previous_points.copy()

                if texture is not None:
                    texture_to_reference = (
                        cv2.getPerspectiveTransform(
                            texture_corners,
                            reference_corners.reshape(4, 2)
                        )
                    )

                tracking_active = True

                print(
                    f"Re-initialization: {len(previous_points)} points"
                )

            except ValueError as error:
                tracking_active = False
                print(error)

        previous_gray = current_gray.copy()

finally:
    cap.release()

    if writer is not None:
        writer.release()
        print(f"Output video closed and saved：{OUTPUT_PATH}")

    cv2.destroyAllWindows()

Save the output video? Enter 'y' to save, any other input to skip: k


Video playback ended.
